# Notion-Zotero Analysis

Full pipeline: load canonical bundles → build summary tables → clean → explore.

**Data sources** (pick one):
-  — bundles pulled via 
-  — bundles pulled via 
-  — bundles parsed from local fixture files

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import re
import ast

from notion_zotero.analysis import (
    load_canonical_records,
    build_summary_dataframes,
    clean_table,
    run_analysis,
    is_accepted,
    task_label_fn,
)
from notion_zotero.analysis.original_db_summary import (
    TYPO_FIXES,
    GENERIC_VALUE_MAP,
    SEARCH_STRATEGY_COLUMNS,
)

accepted_only = True  # Set to True to analyze only accepted papers, False to include all

# Load from live pulled data
pulled_dir = Path("data/pulled/notion/learning_analytics_review")
if not pulled_dir.exists() or not any(pulled_dir.glob("*.canonical.json")):
    raise FileNotFoundError(
        f"No pulled data found in {pulled_dir}. "
        "Run: notion-zotero pull-notion"
    )

bundles = load_canonical_records(str(pulled_dir))
print(f"Loaded {len(bundles)} bundles from {pulled_dir}")

In [ ]:
# global functions for images

# ------------------------------------------------------------
# Systematic-review / modern academic infographic style
# ------------------------------------------------------------

STYLE_COLORS = {
    "background": "#F7F9FA",      # soft off-white slide background
    "panel": "#FFFFFF",           # chart/card background
    "text": "#111827",            # dark navy/charcoal
    "muted_text": "#5B6470",      # secondary labels
    "grid": "#E6EBEF",            # very light gridlines
    "border": "#DDE3E8",

    "teal": "#0B8196",            # identification / primary
    "orange": "#C96B00",          # filtering / warning
    "red": "#B91C1C",             # excluded / negative
    "green": "#16834A",           # included / positive
    "bluegray": "#60738A",        # arrows / neutral accent
    "light_green": "#EAF7EF",
}

INFOGRAPHIC_PALETTE = [
    STYLE_COLORS["teal"],
    STYLE_COLORS["orange"],
    STYLE_COLORS["green"],
    STYLE_COLORS["red"],
    STYLE_COLORS["bluegray"],
    "#8A6F9E",
    "#4C9A7F",
]


def set_infographic_seaborn_style(context="talk"):
    """
    Apply a modern academic infographic style inspired by the reference image.
    Best for presentations, reports, systematic reviews, and thesis charts.
    """

    sns.set_theme(
        context=context,
        style="whitegrid",
        palette=INFOGRAPHIC_PALETTE,
        rc={
            # Figure and axes
            "figure.facecolor": STYLE_COLORS["background"],
            "axes.facecolor": STYLE_COLORS["panel"],
            "savefig.facecolor": STYLE_COLORS["background"],

            # Typography
            "font.family": "sans-serif",
            "font.sans-serif": [
                "Inter",
                "Aptos",
                "Arial",
                "Helvetica",
                "DejaVu Sans",
            ],
            "text.color": STYLE_COLORS["text"],
            "axes.labelcolor": STYLE_COLORS["muted_text"],
            "xtick.color": STYLE_COLORS["muted_text"],
            "ytick.color": STYLE_COLORS["muted_text"],

            # Titles and labels
            "axes.titlesize": 18,
            "axes.titleweight": "bold",
            "axes.labelsize": 12,
            "xtick.labelsize": 11,
            "ytick.labelsize": 11,

            # Lines and grid
            "grid.color": STYLE_COLORS["grid"],
            "grid.linewidth": 0.8,
            "axes.edgecolor": STYLE_COLORS["border"],
            "axes.linewidth": 0.8,

            # Remove visual clutter
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.spines.left": False,

            # Legend
            "legend.frameon": False,
            "legend.fontsize": 11,

            # Output
            "figure.dpi": 120,
            "savefig.dpi": 300,
            "savefig.bbox": "tight",
        },
    )


def polish_infographic_axes(
    ax,
    title=None,
    subtitle=None,
    xlabel=None,
    ylabel=None,
    show_y_grid=True,
):
    """
    Apply final chart-level styling after creating a Seaborn plot.
    """

    ax.set_axisbelow(True)

    if show_y_grid:
        ax.grid(axis="y", color=STYLE_COLORS["grid"], linewidth=0.8)
        ax.grid(axis="x", visible=False)
    else:
        ax.grid(False)

    sns.despine(ax=ax, left=True, bottom=False)

    if title:
        ax.set_title(
            title,
            loc="left",
            pad=22 if subtitle else 12,
            fontsize=18,
            fontweight="bold",
            color=STYLE_COLORS["text"],
        )

    if subtitle:
        ax.text(
            0,
            1.04,
            subtitle,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=11,
            fontstyle="italic",
            color=STYLE_COLORS["muted_text"],
        )

    if xlabel is not None:
        ax.set_xlabel(xlabel, labelpad=10)

    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=10)

    ax.tick_params(axis="both", length=0)

    return ax


def add_bar_labels(ax, fmt="{:.0f}", padding=3):
    """
    Add clean numeric labels above bars.
    """

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt=fmt,
            padding=padding,
            fontsize=11,
            fontweight="bold",
            color=STYLE_COLORS["text"],
        )

set_infographic_seaborn_style()

In [ ]:
# --- Cell 2: Inspect loaded bundles ---
print(f"Loaded {len(bundles)} canonical bundles")

if bundles:
    sample = bundles[0]
    refs = sample.get("references", [])
    if refs:
        r = refs[0]
        print(f"Sample: {r.get("title", "(no title)")} ({r.get("year", "?")})")

In [ ]:
# --- Cell 3: Build summary tables (accepted papers only) ---
if accepted_only is True:
    accepted_bundles = [b for b in bundles if is_accepted(b)]
    print(f"Accepted: {len(accepted_bundles)} / {len(bundles)} total bundles")

else:
    accepted_bundles = bundles
    print(f"Including all bundles (accepted + excluded): {len(accepted_bundles)} total bundles")

dfs = build_summary_dataframes(accepted_bundles, task_label_fn=task_label_fn)
for name, df in dfs.items():
    print(f"  {name}: {len(df)} rows x {len(df.columns)} cols")
dfs.get("Reading List")

In [ ]:
# --- Cell 4: Clean tables ---
# MY_TYPO_FIXES and MY_VALUE_MAP extend the package defaults with notebook-specific overrides.
MY_TYPO_FIXES = {
    **TYPO_FIXES,
    r"\bMAchine\b": "Machine",
    r"Fiiltering": "Filtering",
    r"Exerrcise": "Exercise",
}
MY_VALUE_MAP = {**GENERIC_VALUE_MAP, "n/a": "Not Applicable", "none": "Not Applicable"}

cleaned_dfs = {}
_clean_logs = {}
for name, df in dfs.items():
    cleaned, log = clean_table(
        df,
        typo_fixes=MY_TYPO_FIXES,
        value_map=MY_VALUE_MAP,
        search_strategy_columns=SEARCH_STRATEGY_COLUMNS,
    )
    cleaned_dfs[name] = cleaned
    _clean_logs[name] = log
    print(f"  {name}: {log['text_updates']} cell(s) normalised")

In [ ]:
norm_log = {
    name: {
        "rows_before": len(dfs[name]),
        "rows_after": len(cleaned_dfs[name]),
        "text_updates": _clean_logs[name]["text_updates"],
        "search_strategy_updates": _clean_logs[name].get("search_strategy_updates", 0),
    }
    for name in dfs
}
pd.DataFrame(norm_log).T

In [ ]:
rl = cleaned_dfs.get("Reading List", pd.DataFrame())
if not rl.empty and "year" in rl.columns:
    display(rl["year"].value_counts().sort_index())
if not rl.empty and "journal" in rl.columns:
    display(rl["journal"].dropna().replace("", pd.NA).dropna().value_counts().head(10))
if not rl.empty and "database" in rl.columns:
    display(rl["database"].dropna().replace("", pd.NA).dropna().value_counts().head(10))
if not rl.empty and "journal_quartile" in rl.columns:
    display(rl["journal_quartile"].dropna().replace("", pd.NA).dropna().value_counts())
for name, df in cleaned_dfs.items():
    if name != "Reading List" and not df.empty:
        print(f"\n{name} ({len(df)} rows):")
        display(df.head(3))

## Paper Analysis Outputs

1. Check and count Unique Search Terms

In [ ]:
# #get value_counts of unique search terms from Reading List
# if "Reading List" in cleaned_dfs:
#     search_terms = cleaned_dfs["Reading List"]["search_terms"].dropna().replace("", pd.NA).dropna()
#     print("\nUnique search terms and their counts:")
#     display(search_terms.value_counts())

2. Get Rejection Status - may also need some typo correction and text normalization, similar to Search Terms Column

In [ ]:
#group by Status and Motive For Exclusion and count
reading_list = cleaned_dfs.get("Reading List", pd.DataFrame())
if not reading_list.empty and "Status" in reading_list.columns and "Motive For Exclusion" in reading_list.columns:
    exclusion_counts = (
        reading_list.groupby(["Status", "Motive For Exclusion"])
        .size()
        .reset_index(name="Count")
    )
    print("\nCounts by Status and Motive For Exclusion:")
    display(exclusion_counts)

3. Figure 2

Evolution of Learner Representation over the Years (only consider top 3: Static Aggregate, Dynamic and Sequence). Line and facet grid per task

In [ ]:
#check Learner Representation column for unique values and counts
if not reading_list.empty and "Learner Representation" in reading_list.columns:
    learner_representation_counts = (
        reading_list["Learner Representation"]
        .dropna()
        .replace("", pd.NA)
        .dropna()
        .value_counts()
        .reset_index(name="Count")
    )
    print("\nLearner Representation counts:")
    display(learner_representation_counts)

In [ ]:
TOP_REPRESENTATIONS = ["Static Aggregate", "Dynamic", "Sequence"]

REPRESENTATION_PALETTE = {
    "Static Aggregate": "#0B8196",
    "Dynamic": "#C96B00",
    "Sequence": "#16834A",
}

def parse_list_like_cell(value):
    """
    Safely handles:
    - actual Python lists: ['Static Aggregate', 'Sequence']
    - strings that look like lists: "[Static Aggregate, Sequence]"
    - single strings: "Static Aggregate"
    """

    # Important: check list-like objects before pd.isna()
    if isinstance(value, (list, tuple, set)):
        return [
            str(v).strip().strip("'").strip('"')
            for v in value
            if str(v).strip()
        ]

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "" or text.lower() in {"nan", "none", "n/a", "not applicable"}:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [
                str(v).strip().strip("'").strip('"')
                for v in parsed
                if str(v).strip()
            ]
    except Exception:
        pass

    text = text.strip("[]")

    values = re.split(r"\s*(?:,|;|\||\n)\s*", text)

    return [
        v.strip().strip("'").strip('"')
        for v in values
        if v.strip()
    ]


reading_list = cleaned_dfs["Reading List"].copy()

paper_key_col = "page_id" if "page_id" in reading_list.columns else "id"

chart1_base = reading_list[
    [paper_key_col, "year", "Learner Representation"]
].copy()

chart1_base = chart1_base.rename(columns={paper_key_col: "paper_key"})

chart1_base["year"] = pd.to_numeric(chart1_base["year"], errors="coerce")
chart1_base = chart1_base.dropna(subset=["year", "Learner Representation"])
chart1_base["year"] = chart1_base["year"].astype(int)

chart1_base["representation"] = chart1_base["Learner Representation"].apply(parse_list_like_cell)
chart1_base = chart1_base.explode("representation")

chart1_base["representation"] = chart1_base["representation"].astype(str).str.strip()

chart1_base = chart1_base[
    chart1_base["representation"].isin(TOP_REPRESENTATIONS)
].copy()

chart1_df = (
    chart1_base
    .drop_duplicates(subset=["paper_key", "year", "representation"])
    .groupby(["year", "representation"])
    .size()
    .reset_index(name="n")
)

all_years = range(chart1_df["year"].min(), chart1_df["year"].max() + 1)

full_index = pd.MultiIndex.from_product(
    [all_years, TOP_REPRESENTATIONS],
    names=["year", "representation"]
)

chart1_df = (
    chart1_df
    .set_index(["year", "representation"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5.8))

sns.lineplot(
    data=chart1_df,
    x="year",
    y="n",
    hue="representation",
    hue_order=TOP_REPRESENTATIONS,
    palette=REPRESENTATION_PALETTE,
    marker="o",
    linewidth=2.8,
    markersize=7,
    ax=ax
)

ax.set_title(
    "Learner Representation Usage Over Time",
    loc="left",
    fontsize=18,
    fontweight="bold",
    pad=18
)

ax.set_xlabel("Publication year")
ax.set_ylabel("Number of papers")

ax.grid(axis="y", linewidth=0.8)
ax.grid(axis="x", visible=False)

sns.despine(ax=ax, left=True, bottom=False)

ax.tick_params(axis="both", length=0)
ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

ax.legend(
    title="Learner Representations",
    loc="right",
    bbox_to_anchor=(1.02, 1),
    frameon=False
)

plt.tight_layout()
plt.show()

Figure 3: Same as Figure 2 but divided per task

In [ ]:
TOP_REPRESENTATIONS = ["Static Aggregate", "Dynamic", "Sequence"]
TASK_ORDER = ["PRED", "DESC", "KT", "ERS"]

TASK_TITLE_MAP = {
    "PRED": "Predictive Modeling",
    "DESC": "Descriptive Modeling",
    "KT": "Knowledge Tracing",
    "ERS": "Educational Recommender Systems",
}

REPRESENTATION_PALETTE = {
    "Static Aggregate": "#0B8196",
    "Dynamic": "#C96B00",
    "Sequence": "#16834A",
}

def parse_list_like_cell(value):
    if isinstance(value, (list, tuple, set)):
        return [
            str(v).strip().strip("'").strip('"')
            for v in value
            if str(v).strip()
        ]

    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "" or text.lower() in {"nan", "none", "n/a"}:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [
                str(v).strip().strip("'").strip('"')
                for v in parsed
                if str(v).strip()
            ]
    except Exception:
        pass

    text = text.strip("[]")

    values = re.split(r"\s*(?:,|;|\||\n)\s*", text)

    return [
        v.strip().strip("'").strip('"')
        for v in values
        if v.strip()
    ]


def map_status_to_task(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    text_low = text.lower()

    if not text_low.startswith("accepted for"):
        return None

    if "pred" in text_low or "predictive" in text_low:
        return "PRED"

    if "desc" in text_low or "descriptive" in text_low:
        return "DESC"

    if "kt" in text_low or "knowledge tracing" in text_low:
        return "KT"

    if "ers" in text_low or "recommender" in text_low or "recsys" in text_low:
        return "ERS"

    return None


reading_list = cleaned_dfs["Reading List"].copy()

paper_key_col = "page_id" if "page_id" in reading_list.columns else "id"

chart2_base = reading_list[
    [paper_key_col, "year", "Learner Representation", "Status", "Status_1"]
].copy()

chart2_base = chart2_base.rename(columns={paper_key_col: "paper_key"})

chart2_base["year"] = pd.to_numeric(chart2_base["year"], errors="coerce")
chart2_base = chart2_base.dropna(subset=["year", "Learner Representation"])
chart2_base["year"] = chart2_base["year"].astype(int)

chart2_base["representation"] = chart2_base["Learner Representation"].apply(parse_list_like_cell)
chart2_base = chart2_base.explode("representation")

chart2_base["representation"] = chart2_base["representation"].astype(str).str.strip()

chart2_base = chart2_base[
    chart2_base["representation"].isin(TOP_REPRESENTATIONS)
].copy()

chart2_base["task_values"] = chart2_base.apply(
    lambda row: parse_list_like_cell(row["Status"]) + parse_list_like_cell(row["Status_1"]),
    axis=1
)

chart2_base = chart2_base.explode("task_values")
chart2_base["task_values"] = chart2_base["task_values"].astype(str).str.strip()
chart2_base["task"] = chart2_base["task_values"].apply(map_status_to_task)

chart2_base = chart2_base[
    chart2_base["task"].isin(TASK_ORDER)
].copy()

chart2_df = (
    chart2_base
    .drop_duplicates(subset=["paper_key", "task", "year", "representation"])
    .groupby(["task", "year", "representation"])
    .size()
    .reset_index(name="n")
)

all_years = range(chart2_df["year"].min(), chart2_df["year"].max() + 1)

full_index = pd.MultiIndex.from_product(
    [TASK_ORDER, all_years, TOP_REPRESENTATIONS],
    names=["task", "year", "representation"]
)

chart2_df = (
    chart2_df
    .set_index(["task", "year", "representation"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

chart2_df["task"] = pd.Categorical(
    chart2_df["task"],
    categories=TASK_ORDER,
    ordered=True
)

g = sns.FacetGrid(
    data=chart2_df,
    col="task",
    col_order=TASK_ORDER,
    hue="representation",
    hue_order=TOP_REPRESENTATIONS,
    palette=REPRESENTATION_PALETTE,
    col_wrap=2,
    height=3.8,
    aspect=1.45,
    sharex=True,
    sharey=True
)

g.map_dataframe(
    sns.lineplot,
    x="year",
    y="n",
    marker="o",
    linewidth=2.4,
    markersize=5.5
)

g.figure.set_size_inches(13, 8.5)

# Build custom legend below the full facet grid
handles = []
labels = []
for ax in g.axes.flat:
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        break

for ax, task_code in zip(g.axes.flat, TASK_ORDER):
    ax.grid(axis="y", linewidth=0.8)
    ax.grid(axis="x", visible=False)

    sns.despine(ax=ax, left=True, bottom=False)

    ax.tick_params(axis="both", length=0)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))

    # Remove any existing facet titles (including "task = ...")
    ax.set_title("")
    ax.set_title("", loc="left")
    ax.set_title("", loc="center")
    ax.set_title("", loc="right")

    # Add only the custom centered title
    ax.set_title(
        TASK_TITLE_MAP[task_code],
        loc="center",
        fontsize=15,
        fontweight="bold",
        pad=14
    )

    ax.set_xlabel("Publication year")
    ax.set_ylabel("Number of papers")

g.figure.suptitle(
    "Learner Representations by Task",
    x=0.03,
    y=0.98,
    ha="left",
    fontsize=20,
    fontweight="bold"
)

g.figure.legend(
    handles,
    labels,
    title="Learner Representations",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=3,
    frameon=False
)

plt.subplots_adjust(top=0.88, bottom=0.16, hspace=0.35, wspace=0.18)
plt.show()